In [ ]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from PIL import Image
import numpy as np
import json
from tqdm import tqdm
import viser
import time
from pathlib import Path

from lac.perception.segmentation import SemanticClasses, UnetSegmentation
from lac.slam.semantic_feature_tracker import SemanticFeatureTracker
from lac.slam.frontend import Frontend
from lac.slam.backend import Backend
from lac.util import load_data, load_images, load_stereo_images, get_positions_from_poses
from lac.params import LAC_DATA_PATH
import lac.params as params

%load_ext autoreload
%autoreload 2

In [ ]:
run_name = "2025-05-28_11-59-12"
data_path = LAC_DATA_PATH / "runs" / run_name
initial_pose, lander_pose, poses, imu_data, cam_config, json_data = load_data(data_path)
config = json.load(open("../configs/nine_loops.json"))
print(f"Loaded {len(poses)} poses")
PRESET = json_data["preset"]
MAP_NUM = 1

# 2D trajectory plots

In [ ]:
from lac.utils.plotting import plot_path_2d

In [ ]:
gt_positions = get_positions_from_poses(poses)
backend_state = np.load(Path(data_path) / "backend_state.npz")
slam_poses = np.load(Path(data_path) / "slam_poses.npy")

In [ ]:
slam_positions = get_positions_from_poses(slam_poses)

In [ ]:
fig = plot_path_2d(gt_positions, color="black", name="Ground Truth")
fig = plot_path_2d(slam_positions, fig=fig, color="orange", name="SLAM")
fig.show()

# Maps

In [ ]:
from lac.slam.backend import SemanticPointCloud
from lac.mapping.mapper import process_map
from lac.utils.plotting import plot_height_error, plot_rock_results

In [ ]:
map_path = LAC_DATA_PATH / "maps" / "competition" / f"Moon_Map_{MAP_NUM:02d}_{PRESET}_rep0.dat"
ground_truth_map = np.load(map_path, allow_pickle=True)
agent_map = ground_truth_map.copy()
point_map = SemanticPointCloud.from_file(Path(data_path) / "semantic_points.npz")
agent_map = process_map(point_map, agent_map)

In [ ]:
plot_height_error(agent_map, ground_truth_map)

In [ ]:
plot_rock_results(ground_truth_map, agent_map)